# F04A INWIT — Viewer HTML + Sélecteur Vitesse
## PENTERACT DORN V3 — VIIe Légion

**Rôle** : Afficher `video_render.mp4` dans Colab, choisir la vitesse de lecture, figer le choix vers F04B.

**Entrées** : `F04_INWIT/IN/video_render.mp4` + `F04_INWIT/IN/plan_de_vol.json`  
**Sorties** : `F04_INWIT/OUT/speed_validated.json` + `F04_INWIT/OUT/plan_de_vol.json` (playback_speed mis à jour)

---
### Avant de lancer
1. Monte ton Google Drive (cellule 1)
2. Vérifie que `F04_INWIT/IN/video_render.mp4` existe (produit par F03)
3. Vérifie que `F04_INWIT/IN/plan_de_vol.json` existe (copié depuis F03 OUT via CUSTOS)
4. Lance les cellules 1, 2, 3 dans l'ordre
5. Dans le viewer : sélectionne la vitesse, clique **FIGER LA VITESSE**
6. Lance la cellule 4 (CUSTOS check-in)

In [ ]:
# CELLULE 1 — Montage Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive monté.')

In [ ]:
# CELLULE 2 — Configuration
DRIVE_BASE = '/content/drive/MyDrive/DRIVE_DORN'  # ← adapter si besoin
print(f'Drive base : {DRIVE_BASE}')

In [ ]:
# CELLULE 3 — Lancement du Viewer F04A INWIT
# Le viewer s'affiche en HTML interactif.
# 1. Regarder la vidéo
# 2. Sélectionner une vitesse (0.5x à 2.0x)
# 3. Cliquer FIGER LA VITESSE  →  speed_validated.json est écrit dans OUT/
import shutil, sys
from pathlib import Path

script_src = Path(DRIVE_BASE) / 'F04_INWIT' / 'CODEBASE' / 'drn_f04a_inwit.py'
script_dst = Path('/content/drn_f04a_inwit.py')
shutil.copy2(script_src, script_dst)
print(f'Script copié : {script_dst}')

# Exécution inline (affiche le viewer dans cette cellule)
%run /content/drn_f04a_inwit.py --drive-base {DRIVE_BASE}

In [ ]:
# CELLULE 4 — Transit CRS_CUSTOS (check-out F04)
# À lancer UNIQUEMENT après avoir cliqué FIGER LA VITESSE dans la cellule 3
import shutil, subprocess, sys, json
from pathlib import Path

# Vérification : speed_validated.json doit exister
speed_file = Path(DRIVE_BASE) / 'F04_INWIT' / 'OUT' / 'speed_validated.json'
if not speed_file.exists():
    print('ERREUR : speed_validated.json introuvable.')
    print('→ Retourner à la cellule 3, sélectionner une vitesse et cliquer FIGER LA VITESSE')
else:
    with open(speed_file) as f:
        sv = json.load(f)
    print(f'Vitesse figée : {sv["playback_speed"]}x  (validated={sv["validated"]})')

    custos_src = Path(DRIVE_BASE) / 'CRS_CUSTOS.py'
    shutil.copy2(custos_src, '/content/CRS_CUSTOS.py')

    result = subprocess.run(
        [sys.executable, '/content/CRS_CUSTOS.py',
         '--frigate', 'F04', '--mode', 'check-out', '--drive-base', DRIVE_BASE],
        capture_output=False
    )
    if result.returncode == 0:
        print('\n✓ CRS_CUSTOS — F04 check-out OK — Transit autorisé vers F04B')
        print(f'→ Vitesse : {sv["playback_speed"]}x')
        if sv['playback_speed'] == 1.0:
            print('→ F04B : aucun re-encode (copy direct)')
        else:
            print(f'→ F04B : FFmpeg setpts=PTS/{sv["playback_speed"]} (re-encode)')
    else:
        print('\n✗ CRS_CUSTOS — F04 check-out FAIL')